In [1]:
import Bio
print(Bio.__version__)

1.88


In [2]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd
from Bio.PDB import MMCIFParser, NeighborSearch
from Bio.SeqUtils import seq1

In [3]:
path = Path("data/8B12.cif")
model = MMCIFParser(QUIET=True).get_structure("8b12", str(path))[0]
print("loaded", path)

loaded data/8B12.cif


In [ ]:
chains = {}
for ch in model:
    res = [r for r in ch if r.id[0] == " "]
    seq = "".join(seq1(r.resname) for r in res)
    chains[ch.id] = (res, seq)
    print(ch.id, len(res), res[0].id[1], res[-1].id[1], "motifs:", len(re.findall(r"[IV]TG", seq)))

A 93 6 98 motifs: 0
B 93 6 98 motifs: 0
C 93 6 98 motifs: 0
D 93 6 98 motifs: 0
E 93 6 98 motifs: 0
F 93 6 98 motifs: 0
G 81 1 81 motifs: 0
H 93 6 98 motifs: 0
I 93 6 98 motifs: 0
X 112 712 869 motifs: 5


In [5]:
atoms = [a for ch in model for r in ch if r.id[0] == " " for a in r]
ns = NeighborSearch(atoms)
print(len(atoms), "atoms indexed")

6685 atoms indexed


In [6]:
def motif_contacts(s2_chain, cutoff=4.5, pattern=r"[IV]TG"):
    res, seq = chains[s2_chain]
    rows = []
    for m in re.finditer(pattern, seq):
        for r in res[m.start():m.start() + 3]:
            for a in r:
                for b in ns.search(a.coord, cutoff):
                    rb = b.get_parent()
                    if rb.get_parent().id != s2_chain:
                        rows.append(dict(
                            s2_res=r.id[1], s2_name=r.resname,
                            partner_chain=rb.get_parent().id,
                            partner_res=rb.id[1], partner_name=rb.resname,
                            atom_pair=f"{a.name}-{b.name}", dist=a - b))
    return pd.DataFrame(rows)

In [7]:
res, seq = chains["X"]
frags, cur = [], [res[0]]
for r in res[1:]:
    if r.id[1] == cur[-1].id[1] + 1:
        cur.append(r)
    else:
        frags.append(cur)
        cur = [r]
frags.append(cur)

for f in frags:
    s = "".join(seq1(r.resname) for r in f)
    print(f[0].id[1], f[-1].id[1], s)
    for m in re.finditer(r"[ILV]TG", s):
        print("   motif", s[m.start():m.end()], "starts at residue", f[m.start()].id[1])

712 731 RITGPGMLATGLITGTPEFR
   motif ITG starts at residue 713
   motif ITG starts at residue 724
773 823 LVCAPRSDQMDRVSGEGKERCHITGDDWSVNKHITGTAGQWASGRNPSMRG
   motif ITG starts at residue 795
   motif ITG starts at residue 806
829 869 ETSAFANRNVPKPEKPGSKITGSSGNDTQGSLITYSGGARG
   motif ITG starts at residue 848


In [8]:
for cid in ["A", "G"]:
    print(cid, chains[cid][1])
print([(r.id[1], r.resname) for r in chains["A"][0] if r.id[1] in (75, 79)])

A GIALGMIETRGLVPAIEAADAMTKAAEVRLVGRQFVGGGYVTVLVRGETGAVNAAVRAGADACERVGDGLVAAHIIARVHSEVENILPKAPQA
G MKIMQVEKTLVSTNRIADMGHKPLLVVWEKPGAPRQVAVDAIGCIPGDWVLCVGSSAAREAAGSKSYPSDLTIIGIIDQWN
[(75, 'LEU'), (79, 'HIS')]


In [10]:
df = motif_contacts("X")
pairs = (df.groupby(["s2_res", "s2_name", "partner_chain", "partner_res", "partner_name"])["dist"]
           .min().reset_index().sort_values("dist"))
pairs.head(30)

,s2_res,s2_name,partner_chain,partner_res,partner_name,dist
20,807,THR,I,79,HIS,2.551898
6,725,THR,A,79,HIS,2.602208
5,725,THR,A,78,ALA,2.740215
19,807,THR,I,78,ALA,2.857181
7,725,THR,A,80,ILE,2.858980
21,807,THR,I,80,ILE,2.934585
11,726,GLY,A,80,ILE,3.230603
15,806,ILE,I,78,ALA,3.237702
3,724,ILE,A,78,ALA,3.337779
22,808,GLY,I,79,HIS,3.395000


residue 79 is Histidine 
residue 75 is Leuicine 
This table takes a CsoS2 residue and gives you the closest extramolecular partner residue to it 

In [13]:
df_all = motif_contacts("X", pattern=r"[ILV]TG")

best = df_all.loc[
    df_all.groupby(["s2_res", "partner_chain", "partner_res"])["dist"].idxmin()
].sort_values("dist")

best[["s2_res", "s2_name", "partner_chain", "partner_res", "partner_name", "atom_pair", "dist"]].head(15)

,s2_res,s2_name,partner_chain,partner_res,partner_name,atom_pair,dist
139,807,THR,I,79,HIS,OG1-ND1,2.551898
55,725,THR,A,79,HIS,OG1-ND1,2.602208
22,725,THR,A,78,ALA,N-O,2.740215
103,807,THR,I,78,ALA,N-O,2.857181
38,725,THR,A,80,ILE,O-N,2.858980
121,807,THR,I,80,ILE,O-N,2.934585
67,726,GLY,A,80,ILE,CA-O,3.230603
80,806,ILE,I,78,ALA,CA-O,3.237702
3,724,ILE,A,78,ALA,CA-O,3.337779
153,808,GLY,I,79,HIS,CA-CE1,3.395000


In [12]:
BACKBONE = {"N", "CA", "C", "O", "OXT"}

parts = df_all.atom_pair.str.split("-", expand=True)
df_all["s2_atom"] = parts[0]
df_all["partner_atom"] = parts[1]
df_all["s2_part"] = df_all.s2_atom.apply(lambda a: "backbone" if a in BACKBONE else "sidechain")
df_all["partner_part"] = df_all.partner_atom.apply(lambda a: "backbone" if a in BACKBONE else "sidechain")

polar = df_all.s2_atom.str[0].isin(["N", "O"]) & df_all.partner_atom.str[0].isin(["N", "O"])
hb = df_all[polar & (df_all.dist < 3.5)].sort_values("dist")
hb[["s2_res", "s2_name", "s2_atom", "s2_part", "partner_chain", "partner_res", "partner_name", "partner_atom", "partner_part", "dist"]]

,s2_res,s2_name,s2_atom,s2_part,partner_chain,partner_res,partner_name,partner_atom,partner_part,dist
139,807,THR,OG1,sidechain,I,79,HIS,ND1,sidechain,2.551898
55,725,THR,OG1,sidechain,A,79,HIS,ND1,sidechain,2.602208
22,725,THR,N,backbone,A,78,ALA,O,backbone,2.740215
103,807,THR,N,backbone,I,78,ALA,O,backbone,2.857181
38,725,THR,O,backbone,A,80,ILE,N,backbone,2.858980
121,807,THR,O,backbone,I,80,ILE,N,backbone,2.934585
126,807,THR,O,backbone,I,79,HIS,ND1,sidechain,3.176719
58,725,THR,OG1,sidechain,A,78,ALA,O,backbone,3.187913
43,725,THR,O,backbone,A,79,HIS,ND1,sidechain,3.366117
144,807,THR,OG1,sidechain,I,78,ALA,O,backbone,3.401874
